# Optuna seul (PyTorch) sur MNIST

Ce notebook réalise l'optimisation d'hyperparamètres avec Optuna seul (sans Ray Tune), dans les **mêmes
conditions expérimentales** que le notebook Ray Tune (même budget, même espace de recherche, même
découpage des données, même prétraitement), pour que les résultats puissent être comparés.
La seule différence qui subsiste est le framework : PyTorch ici, TensorFlow dans le notebook Ray Tune.


In [3]:
%pip install torch torchvision --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: C:\Users\HP\Documents\TP_RayTune\.venv\Scripts\python.exe -m pip install --upgrade pip


In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
import optuna

torch.set_num_threads(2)   # comparable aux 2 CPU alloués à Ray Tune
optuna.logging.set_verbosity(optuna.logging.WARNING)   # on affiche nos propres lignes de suivi

ModuleNotFoundError: No module named 'torch'

## Paramètres de l'expérience (identiques à ceux du notebook Ray Tune)

In [ ]:
N_TRIALS = 8     # = NUM_SAMPLES du notebook Ray Tune
EPOCHS = 10      # époques par essai
N_SEEDS = 3      # graines pour l'évaluation finale sur le test
SEED = 42        # graine pour la recherche

## Chargement du dataset

In [ ]:
import os

chemin = os.path.expanduser("~/.keras/datasets/mnist.npz")
d = np.load(chemin)

X_all = torch.tensor(d["x_train"]).float().div(255.0)
y_all = torch.tensor(d["y_train"]).long()
X_test = torch.tensor(d["x_test"]).float().div(255.0)
y_test = torch.tensor(d["y_test"]).long()

n_train = int(0.9 * len(X_all))
X_tr, y_tr = X_all[:n_train], y_all[:n_train]
X_val, y_val = X_all[n_train:], y_all[n_train:]

print("Entraînement :", tuple(X_tr.shape), "| Validation :", tuple(X_val.shape), "| Test :", tuple(X_test.shape))

## Modèle MLP et entraînement

Même architecture que dans le notebook TensorFlow : une couche cachée ReLU, un dropout, une sortie à 10 classes.
Les données sont gardées en mémoire sous forme de tenseurs et découpées en lots à la main, ce qui est beaucoup
plus rapide qu'un `DataLoader` avec transformations image par image.

In [ ]:
def construire_modele(units, dropout):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(28 * 28, units),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(units, 10)
    )


def precision(modele, X, y):
    """Accuracy en mode évaluation (dropout désactivé)."""
    modele.eval()
    with torch.no_grad():
        predictions = modele(X).argmax(dim=1)
    return (predictions == y).float().mean().item()


def entrainer(params, epochs=EPOCHS, seed=SEED):
    torch.manual_seed(seed)
    modele = construire_modele(params["units"], params["dropout"])
    optimiseur = torch.optim.Adam(modele.parameters(), lr=params["learning_rate"])
    perte = nn.CrossEntropyLoss()
    taille_lot = params["batch_size"]
    n = X_tr.shape[0]

    acc_val = None
    for epoch in range(epochs):
        modele.train()                       # dropout actif pendant l'entraînement
        ordre = torch.randperm(n)
        for i in range(0, n, taille_lot):
            idx = ordre[i:i + taille_lot]
            optimiseur.zero_grad()
            loss = perte(modele(X_tr[idx]), y_tr[idx])
            loss.backward()
            optimiseur.step()
        acc_val = precision(modele, X_val, y_val)   # modele.eval() appelé dans precision
    return modele, acc_val

## La fonction objectif d'Optuna

In [ ]:
def objective(trial):
    params = {
        "units": trial.suggest_categorical("units", [32, 64, 128, 256, 512]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
    }
    _, acc_val = entrainer(params)
    return acc_val     # accuracy de validation à la dernière époque, comme dans Ray Tune

## L'optimisation

In [ ]:
def print_callback(study, trial):
    print(f"Essai {trial.number} terminé : accuracy validation = {trial.value:.4f} | params = {trial.params}")


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

t0 = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_callback])
duree_totale = time.perf_counter() - t0

print()
print("Meilleurs hyperparamètres :", study.best_params)
print(f"Meilleure accuracy de validation : {study.best_value:.2%}")
print(f"Durée totale de la recherche : {duree_totale:.0f} s")

## Convergence

Meilleure accuracy de validation atteinte au fil des essais.

In [ ]:
df_essais = study.trials_dataframe().sort_values("number")

plt.figure(figsize=(8, 4))
plt.plot(df_essais["number"] + 1, df_essais["value"], "o", alpha=0.5, label="Accuracy de l'essai")
plt.plot(df_essais["number"] + 1, df_essais["value"].cummax(), "-o", label="Meilleure accuracy cumulée")
plt.xlabel("Essai")
plt.ylabel("Accuracy de validation")
plt.title("Convergence d'Optuna (PyTorch)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Importance des hyperparamètres

Ici l'objet `study` existe, donc on peut calculer l'importance statistique avec `optuna.importance`.
Avec un petit nombre d'essais, ces valeurs sont très instables : à interpréter avec prudence.

In [ ]:
try:
    from optuna.importance import get_param_importances

    importances = get_param_importances(study)
    noms = list(importances.keys())
    valeurs = list(importances.values())

    plt.figure(figsize=(7, 3.5))
    plt.barh(noms[::-1], valeurs[::-1])
    plt.xlabel("Importance estimée")
    plt.title(f"Importance des hyperparamètres (Optuna, {len(study.trials)} essais)")
    plt.tight_layout()
    plt.show()
except Exception as erreur:
    print("Importance non calculable avec ces essais :", erreur)

## Évaluation finale sur le jeu de test

La meilleure configuration est réentraînée avec plusieurs graines ; on rapporte la moyenne et l'écart-type
de l'accuracy de test (jeu de test jamais utilisé pendant la recherche).

In [ ]:
scores = []
for graine in range(N_SEEDS):
    modele, _ = entrainer(study.best_params, seed=100 + graine)
    scores.append(precision(modele, X_test, y_test) * 100)
scores = np.array(scores)

print(f"Accuracy de test : {scores.mean():.2f} % ± {scores.std():.2f} (sur {N_SEEDS} graines)")

## Résumé 

In [ ]:
resume = pd.DataFrame([{
    "Méthode": "Optuna seul (PyTorch)",
    "Essais": len(study.trials),
    "Accuracy validation (%)": round(study.best_value * 100, 2),
    "Durée totale recherche (s)": round(duree_totale, 1),
    "Unités": study.best_params["units"],
    "Dropout": round(study.best_params["dropout"], 3),
    "Learning rate": study.best_params["learning_rate"],
    "Batch size": study.best_params["batch_size"],
    "Accuracy test moyenne (%)": round(float(scores.mean()), 2),
    "Écart-type test (%)": round(float(scores.std()), 2),
}])
display(resume)

resume.to_csv("resultats_optuna_pytorch.csv", index=False)
print("Résumé enregistré dans resultats_optuna_pytorch.csv")